In [1]:
import torch
import torch.nn as nn

In [2]:
batch_size = 64
sequence_len = 64
embedding_dim = 32
layer_size = 256
steps = 5000

In [3]:
# input & output initialization
with open("inputs-shakespeare.txt", "r") as f:
    data = f.read()
chars = sorted(set(data))
vocab_size = len(chars)

ctoi = {c: i for i, c in enumerate(chars)}
itoc = {i: c for i, c in enumerate(chars)}

train = data[:int(len(data) * 0.9)]
val = data[int(len(data) * 0.9):]

x_train = torch.tensor([ctoi[ch] for ch in train[:-1]], dtype=torch.long).unfold(0, sequence_len, 1)
y_train = torch.tensor([ctoi[ch] for ch in train[1:]], dtype=torch.long).unfold(0, sequence_len, 1)

x_val = torch.tensor([ctoi[ch] for ch in val[:-1]], dtype=torch.long).unfold(0, sequence_len, sequence_len)
y_val = torch.tensor([ctoi[ch] for ch in val[1:]], dtype=torch.long).unfold(0, sequence_len, sequence_len)

In [4]:
# model declaration
class BagOfWords(nn.Module):
    def __init__(self, sequence_len, vocab_size, embedding_dim, layer_size):
        super().__init__()
        
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        # self.causal_mean

        self.head = nn.Sequential(
            nn.Linear(in_features=embedding_dim, out_features=layer_size),
            nn.ReLU(),
            nn.Linear(in_features=layer_size, out_features=vocab_size),
        )     

        mask = torch.zeros(sequence_len, sequence_len)
        mask = mask.masked_fill(torch.tril(torch.ones(sequence_len, sequence_len)) == 0, -torch.inf)
        self.register_buffer('mask', mask, persistent=True)

    def causal_mean(self, input):
        mask = self.mask[:input.size(1), :input.size(1)]                
        return torch.softmax(mask, 1) @ input
        
    def forward(self, input):        
        e = self.embedding(input)
        c = self.causal_mean(e)
        logits = self.head(c)
        return logits

In [5]:
# model initialization
model = BagOfWords(sequence_len, vocab_size, embedding_dim, layer_size)
# optimizer initialization
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)

In [6]:
# training
for step in range(steps):
    optimizer.zero_grad()
    
    ix = torch.randint(0, len(x_train), (batch_size,))
    logits_train = model(x_train[ix])
    loss_train = nn.functional.cross_entropy(logits_train.view(-1, vocab_size), y_train[ix].view(-1))

    if step % 1000 == 0 or step == steps - 1:
        with torch.no_grad():
            logits_val = model(x_val)
            loss_val = nn.functional.cross_entropy(logits_val.view(-1, vocab_size), y_val.view(-1))
            print(f"Step {step}: train loss = {loss_train.item()}, val loss = {loss_val.item()}")

    loss_train.backward()
    optimizer.step()

Step 0: train loss = 4.191404342651367, val loss = 4.191176891326904
Step 1000: train loss = 3.0615086555480957, val loss = 3.0647079944610596
Step 2000: train loss = 3.0106799602508545, val loss = 3.0304925441741943
Step 3000: train loss = 2.9768710136413574, val loss = 3.0234763622283936
Step 4000: train loss = 2.9480838775634766, val loss = 3.009993314743042
Step 4999: train loss = 2.9504685401916504, val loss = 3.0113942623138428


In [7]:
# prediction
with torch.inference_mode():
    output = ['R', 'O', 'M', 'E', 'O', ':', '\n']
    max_output_length = 1000

    i = torch.tensor([[ctoi[ch] for ch in output]], dtype=torch.long)            

    while len(output) < max_output_length:
        logits = model(i)

        P = torch.softmax(logits[0, -1, :], dim=0)
        j = torch.multinomial(P, num_samples=1, replacement=True).item()
        output.append(itoc[j])

        i = torch.tensor([[ctoi[ch] for ch in output[-sequence_len:]]], dtype=torch.long)

In [8]:
print(''.join(output))

ROMEO:
oso  ud,b am cftrkei, unovtosh e;eamh  teanr i hdi wtetse rfe ddmt
hgein ieel afreDnaro nvtob ntihhbei,e  cl l  senhgfavoerg yeaweEau s eir.ne ra
rmpt nsi eeyl 
e
KRhiAodUlJU ENRR:,oIT  a! ndbcaA
IMtteCT:hiyearesc,n ro,std  nc e hl 
adwi eIhacpeoe o
h,ek tifoem wlf osyu bW fstehssihipeeFo r.reollr
 u c omt lrrle erareoocwhee?asute 
N kvl
hRoIfllIvOeIa:yao d G o
Lsuiee ydv e,fel  her  dv Lhei oiaaannmruine!rgr  bsp.yrryosts  ce
 poedattPaonl sbihssh, do
A  tooo!,aT
w yauPhnggEIt BE EmYr ShG ,r tIhompiaenihvd:ci m' uoaa:
s niettbopnod orfeboiooiotnstn miedsn h eT otitn,'
wnn e swsofbdho bsaupntsget:.o
ftre o ao
eRrsh shTm hU'edoamiHdbsmysa rdrhthait  c G Ip kee itasftis  o; oo;p sms eah
iis,lbew i alnae tndierwrnhps meWeno
g fntsifta  oihdmTgge s trhlghraa aetsf e opso.oeiid , ndwhe i
 neaaes nn ms sla nefess,of  eE'o aydY
eoystr 
Nci eYpwblusfi urk iobri ofhor : e r iathlltoa crae   TIalpm hm o:
groeo  lawhnn ahm.a rls
nteaEeoslosr
eg ah  L rse
aIVyNlUEYh JaUOLA::
i;vet bl ntiI a